In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

from collections import OrderedDict

H = int
W = int
C = int

In [27]:
class HeadFeatureExtractor(nn.Module):
    def __init__(self, in_channels, repeats = 2, activation = nn.LeakyReLU(0.1)):
        super().__init__()
        self.extractor = nn.ModuleList([nn.Sequential(nn.Conv2d(in_channels, in_channels, 3, padding=1), 
                                        nn.BatchNorm2d(in_channels),
                                        activation) 
                                        for _ in range(repeats)])
    def forward(self, x):
        for block in self.extractor:
            x = block(x)
        return x


class ClassificationHead(nn.Module):
    def __init__(self, in_channels = 256, num_classes = 10, activation = nn.LeakyReLU(0.1)):
        super().__init__()
        self.extractor = HeadFeatureExtractor(256, 2, activation)
        self.conv = nn.Conv2d(in_channels, num_classes, 1)
    
    def forward(self, x):
        x = self.extractor(x)
        x = self.conv(x)

        return nn.functional.sigmoid(x)


class RegressionHead(nn.Module):
    def __init__(self, in_channels = 256, activation = nn.LeakyReLU(0.1)):
        super().__init__()
        self.extractor = HeadFeatureExtractor(in_channels, 2, activation)
        self.conv_object = nn.Conv2d(in_channels, 1, 1)
        self.conv_bbox = nn.Conv2d(in_channels, 4, 1)

    def forward(self, x):
        x = self.extractor(x)
        x_o = self.conv_object(x)
        x_b = self.conv_bbox(x)

        return x_o, x_b
    
class YoloXHead(nn.Module):
    def __init__(self, in_channels = 256, num_classes = 10, activation = nn.LeakyReLU(0.1)):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels, 1)
        self.bn = nn.BatchNorm2d(in_channels)
        self.activation = activation
        self.regression_head = RegressionHead(in_channels, activation)
        self.classification_head = ClassificationHead(in_channels, num_classes, activation)

    def forward(self, x):
        x = self.activation(self.bn(self.conv(x)))
        cls_out = self.classification_head(x)
        reg_out = self.regression_head(x)

        return cls_out, reg_out[0], reg_out[1]



In [28]:
sample = torch.rand((1, 256, 128, 128))
head = YoloXHead(256, 10, nn.SiLU())
out = head(sample)
for o in out:
    print(o.size())

torch.Size([1, 10, 128, 128])
torch.Size([1, 1, 128, 128])
torch.Size([1, 4, 128, 128])


In [ ]:

class DarkNet53ResidualBlock(nn.Module):
    def __init__(self, channel_dims: list[C], activation = nn.LeakyReLU(0.1)):
        super().__init__()
        self.activation = activation
        self.conv1 = nn.Conv2d(channel_dims[0], channel_dims[1], 1)
        self.bnorm1 = nn.BatchNorm2d(channel_dims[1])

        self.conv2 = nn.Conv2d(channel_dims[1], channel_dims[2], 3, padding=1)
        self.bnorm2 = nn.BatchNorm2d(channel_dims[2])
        
    def forward(self, input):
        x = self.activation(self.bnorm1(self.conv1(input)))
        x = self.activation(self.bnorm2(self.conv2(x)))
        return x + input
    

class DarkNet53(nn.Module):
    def __init__(self, activation = nn.LeakyReLU(0.1)):
        super().__init__()

        self.activation = activation
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, 2, padding=1)
        self.bnorm2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, 2, padding=1)
        self.bnorm3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, 2, padding=1)
        self.bnorm4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, 3, 2, padding=1)
        self.bnorm5 = nn.BatchNorm2d(512)
        self.conv6 = nn.Conv2d(512, 1024, 3, 2, padding=1)
        self.bnorm6 = nn.BatchNorm2d(1024)

        self.darknet_block1 = DarkNet53ResidualBlock([64, 32, 64])
        self.darknet_block2 = nn.ModuleList([DarkNet53ResidualBlock([128, 64, 128], activation) for _ in range(2)])
        self.darknet_block3 = nn.ModuleList([DarkNet53ResidualBlock([256, 128, 256], activation) for _ in range(8)])
        self.darknet_block4 = nn.ModuleList([DarkNet53ResidualBlock([512, 256, 512], activation) for _ in range(8)])
        self.darknet_block5 = nn.ModuleList([DarkNet53ResidualBlock([1024, 512, 1024], activation) for _ in range(4)])

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.dense = nn.Linear(1024, 1000)
        self.softmax = nn.Softmax()

    def forward(self, input):
        x = self.activation(self.bnorm1(self.conv1(input)))
        x = self.activation(self.bnorm2(self.conv2(x)))

        x = self.darknet_block1(x)
        x = self.activation(self.bnorm3(self.conv3(x)))

        for darknet_block in self.darknet_block2:
            x = darknet_block(x)
        x = self.activation(self.bnorm4(self.conv4(x)))

        for darknet_block in self.darknet_block3:
            x = darknet_block(x)
        x = self.activation(self.bnorm5(self.conv5(x)))

        for darknet_block in self.darknet_block4:
            x = darknet_block(x)
        x = self.activation(self.bnorm6(self.conv6(x)))

        for darknet_block in self.darknet_block5:
            x = darknet_block(x)

        x = torch.flatten(self.avgpool(x),1)
        print(x.size)
        x = self.dense(x)
        x = self.softmax(x)
        return x
    


In [3]:
test = DarkNet53()
sample = torch.rand((1, 3, 256, 256))
test(sample).size()

<built-in method size of Tensor object at 0x768efc316a80>


/home/shashankg/packages/YOLOX/yolo/lib/python3.12/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


torch.Size([1, 1000])

In [95]:
print([DarkNet53ResidualBlock([128, 64, 128])]*2)

[DarkNet53ResidualBlock(
  (lrelu): LeakyReLU(negative_slope=0.01)
  (conv1): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
), DarkNet53ResidualBlock(
  (lrelu): LeakyReLU(negative_slope=0.01)
  (conv1): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)]


In [ ]:
Ordered